# 01 - Exploring Neo4j Knowledge Graph

This notebook introduces the Smart Home Knowledge Graph stored in Neo4j.

## Learning Objectives

1. Understand how to connect to Neo4j from Python
2. Explore the graph schema (nodes and relationships)
3. Write basic Cypher queries
4. Visualize the graph structure

## Prerequisites

- Neo4j running (locally or Aura cloud)
- Seed data loaded (`data/seed_graph.cypher`)
- `.env` file configured with credentials

## Setup

In [ ]:
# Add parent directory to path for imports
import sys
sys.path.insert(0, '..')

# Load environment variables
from dotenv import load_dotenv
load_dotenv('../.env')

# Import our modules
from src.graph.connection import Neo4jConnection, get_connection
from src.graph.queries import SmartHomeQueries

print("✓ Imports successful!")

✓ Imports successful!


## 1. Connecting to Neo4j

The `Neo4jConnection` class manages our database connection.

In [ ]:
# Create connection
conn = Neo4jConnection()
conn.connect()

# Health check
health = conn.health_check()
print("Database Health Check")
print("=" * 40)
for key, value in health.items():
    print(f"  {key}: {value}")

Database Health Check
  status: healthy
  database: neo4j
  name: Neo4j Kernel
  version: 2025.12.1
  edition: community
  node_count: 33
  relationship_count: 68


### ⚠️ If Connection Failed

Make sure Neo4j is running:

**Option 1: Docker (Recommended)**
```bash
docker run \
  --name neo4j-smarthome \
  -p 7474:7474 -p 7687:7687 \
  -e NEO4J_AUTH=neo4j/password \
  -d neo4j:latest
```

**Option 2: Neo4j Aura (Cloud)**
1. Go to https://neo4j.com/cloud/aura/
2. Create a free instance
3. Copy credentials to `.env` file

## 2. Loading Seed Data

If your database is empty, let's load the seed data.

In [ ]:
# Check if data exists
result = conn.query_single("MATCH (n) RETURN count(n) AS count")
node_count = result['count'] if result else 0

print(f"Current node count: {node_count}")

if node_count == 0:
    print("\n⚠️ Database is empty!")
    print("Loading seed data from data/seed_graph.cypher...")
    
    # Load seed data automatically
    import os
    seed_file = os.path.join('..', 'data', 'seed_graph.cypher')
    
    if os.path.exists(seed_file):
        result = conn.run_cypher_file(seed_file)
        print(f"\n✓ Seed data loaded!")
        print(f"  - Executed {result['executed']} statements")
        if result['errors']:
            print(f"  - Errors: {len(result['errors'])}")
            for err in result['errors']:
                print(f"    {err['error']}")
        
        # Verify data was loaded
        new_count = conn.query_single("MATCH (n) RETURN count(n) AS count")
        print(f"\n✓ New node count: {new_count['count']}")
    else:
        print(f"\n❌ Seed file not found at: {seed_file}")
        print("Please load the seed data manually:")
        print("  1. Open Neo4j Browser at http://localhost:7474")
        print("  2. Copy contents of data/seed_graph.cypher")
        print("  3. Paste and run in the query editor")
else:
    print("\n✓ Database has data!")

Current node count: 33

✓ Database has data!


## 3. Understanding the Graph Schema

Let's explore what's in our knowledge graph.

In [ ]:
# Get node labels and counts
schema_query = """
CALL db.labels() YIELD label
CALL {
    WITH label
    MATCH (n) WHERE label IN labels(n)
    RETURN count(n) AS count
}
RETURN label, count
ORDER BY count DESC
"""

labels = conn.query(schema_query)
print("Node Labels (Entity Types)")
print("=" * 40)
for row in labels:
    print(f"  {row['label']}: {row['count']} nodes")

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (label) { ... }', position=<SummaryInputPosition line=3, column=1, offset=30>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 30, 'line': 3, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nCALL db.labels() YIELD label\nCALL {\n    WITH label\n    MATCH (n) WHERE label IN labels(n)\n    RETURN count(n) AS count\n}\nRETURN label, count\nORDER BY count DESC\n'


Node Labels (Entity Types)
  Device: 13 nodes
  Capability: 10 nodes
  Room: 5 nodes
  Scene: 5 nodes


In [ ]:
# Get relationship types
rel_query = """
CALL db.relationshipTypes() YIELD relationshipType
CALL {
    WITH relationshipType
    MATCH ()-[r]->() WHERE type(r) = relationshipType
    RETURN count(r) AS count
}
RETURN relationshipType, count
ORDER BY count DESC
"""

relationships = conn.query(rel_query)
print("Relationship Types")
print("=" * 40)
for row in relationships:
    print(f"  {row['relationshipType']}: {row['count']} edges")

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (relationshipType) { ... }', position=<SummaryInputPosition line=3, column=1, offset=52>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 52, 'line': 3, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nCALL db.relationshipTypes() YIELD relationshipType\nCALL {\n    WITH relationshipType\n    MATCH ()-[r]->() WHERE type(r) = relationshipType\n    RETURN count(r) AS count\n}\nRETURN relationshipType, count\nORDER BY count DESC\n'


Relationship Types
  HAS_CAPABILITY: 33 edges
  CONTAINS: 13 edges
  USES_DEVICE: 11 edges
  ADJACENT_TO: 6 edges
  APPLIES_TO: 5 edges


### Teaching Point: Graph Schema

Our smart home graph has:

**Node Types (Entities)**
- `Room` - Physical spaces in the home
- `Device` - Smart devices (lights, TV, speakers)
- `Capability` - What devices can do (dim, play_music)
- `Scene` - Predefined configurations (Movie Night)

**Relationship Types**
- `CONTAINS` - Room contains Device
- `HAS_CAPABILITY` - Device has Capability
- `APPLIES_TO` - Scene applies to Room
- `USES_DEVICE` - Scene uses Device
- `ADJACENT_TO` - Room is adjacent to Room

## 4. Basic Cypher Queries

Cypher is Neo4j's query language. Let's learn the basics.

In [ ]:
# Query 1: Get all rooms
rooms = conn.query("""
    MATCH (r:Room)
    RETURN r.name AS name, r.floor AS floor, r.type AS type
    ORDER BY r.floor, r.name
""")

print("All Rooms in the Smart Home")
print("=" * 40)
for room in rooms:
    print(f"  {room['name']} ({room['type']}) - {room['floor']}")

All Rooms in the Smart Home
  Bathroom (utility) - First Floor
  Home Office (work) - First Floor
  Master Bedroom (bedroom) - First Floor
  Kitchen (utility) - Ground Floor
  Living Room (entertainment) - Ground Floor


In [ ]:
# Query 2: Get devices in a specific room
# Note: We use parameters ($room_name) instead of string formatting!

room_name = "Living Room"

devices = conn.query("""
    MATCH (r:Room {name: $room_name})-[:CONTAINS]->(d:Device)
    RETURN d.name AS name, d.device_type AS type, d.brand AS brand
    ORDER BY d.name
""", {"room_name": room_name})

print(f"Devices in {room_name}")
print("=" * 40)
for device in devices:
    print(f"  {device['name']} ({device['type']}) - {device['brand']}")

Devices in Living Room
  Ceiling Light (light) - Philips Hue
  Floor Lamp (light) - IKEA Tradfri
  Smart Speaker (speaker) - Sonos
  Smart TV (television) - Samsung
  Thermostat (thermostat) - Nest
  Window Blinds (blinds) - Lutron


In [ ]:
# Query 3: Follow relationships - Room -> Device -> Capability
# This is a 2-hop traversal!

capabilities = conn.query("""
    MATCH (r:Room {name: $room_name})-[:CONTAINS]->(d:Device)-[:HAS_CAPABILITY]->(c:Capability)
    RETURN d.name AS device, collect(c.name) AS capabilities
    ORDER BY d.name
""", {"room_name": "Living Room"})

print("Device Capabilities in Living Room")
print("=" * 40)
for row in capabilities:
    caps = ", ".join(row['capabilities'])
    print(f"  {row['device']}: {caps}")

Device Capabilities in Living Room
  Ceiling Light: color, dim, power
  Floor Lamp: dim, power
  Smart Speaker: announce, play_music, volume, power
  Smart TV: input_select, volume, power
  Thermostat: temperature
  Window Blinds: open_close


### Teaching Point: Why Graphs?

Compare this to how you'd do it with SQL:

```sql
-- SQL equivalent (multiple JOINs needed)
SELECT d.name, GROUP_CONCAT(c.name)
FROM rooms r
JOIN room_devices rd ON r.id = rd.room_id
JOIN devices d ON rd.device_id = d.id
JOIN device_capabilities dc ON d.id = dc.device_id
JOIN capabilities c ON dc.capability_id = c.id
WHERE r.name = 'Living Room'
GROUP BY d.id;
```

Cypher is more intuitive for relationship-heavy queries!

In [ ]:
# Query 4: Find devices by capability
# "What devices can dim?"

dimmable = conn.query("""
    MATCH (r:Room)-[:CONTAINS]->(d:Device)-[:HAS_CAPABILITY]->(c:Capability {name: 'dim'})
    RETURN r.name AS room, d.name AS device
    ORDER BY r.name
""")

print("All Dimmable Devices")
print("=" * 40)
for row in dimmable:
    print(f"  {row['room']}: {row['device']}")

All Dimmable Devices
  Home Office: Desk Lamp
  Kitchen: Kitchen Light
  Living Room: Floor Lamp
  Living Room: Ceiling Light
  Master Bedroom: Bedside Lamp
  Master Bedroom: Bedroom Light


In [ ]:
# Query 5: Get scene details with associated devices

scenes = conn.query("""
    MATCH (s:Scene)-[:APPLIES_TO]->(r:Room)
    OPTIONAL MATCH (s)-[uses:USES_DEVICE]->(d:Device)
    RETURN s.name AS scene,
           s.mood AS mood,
           r.name AS room,
           collect({device: d.name, action: uses.action}) AS device_actions
""")

print("Available Scenes")
print("=" * 40)
for scene in scenes:
    print(f"\n{scene['scene']} (mood: {scene['mood']})")
    print(f"  Room: {scene['room']}")
    print("  Actions:")
    for action in scene['device_actions']:
        if action['device']:
            print(f"    - {action['device']}: {action['action']}")

Available Scenes

Movie Night (mood: relaxed)
  Room: Living Room
  Actions:
    - Window Blinds: close
    - Floor Lamp: dim to 30%
    - Ceiling Light: dim to 20%, warm color
    - Smart TV: power on

Good Morning (mood: energetic)
  Room: Master Bedroom
  Actions:
    - Bedroom Speaker: play morning playlist
    - Bedroom Light: bright white light

Bedtime (mood: calm)
  Room: Master Bedroom
  Actions:
    - Bedside Lamp: warm orange, dim
    - Bedroom Light: dim to 10%

Focus Mode (mood: focused)
  Room: Home Office
  Actions:
    - Desk Lamp: bright, cool white

Party Mode (mood: exciting)
  Room: Living Room
  Actions:
    - Smart Speaker: play party playlist, high volume
    - Ceiling Light: colorful cycling


## 5. Using Our Query Templates

The `SmartHomeQueries` class provides pre-built queries.

In [ ]:
queries = SmartHomeQueries()

# Get all available query methods
methods = [m for m in dir(queries) if not m.startswith('_')]
print("Available Query Templates:")
print("=" * 40)
for method in methods:
    print(f"  - {method}()")

Available Query Templates:
  - clear_all_data()
  - get_all_capabilities()
  - get_all_devices()
  - get_all_rooms()
  - get_all_scenes()
  - get_context_for_capability_action()
  - get_context_for_multi_room()
  - get_context_for_room_action()
  - get_device_details()
  - get_devices_by_type()
  - get_devices_with_capability()
  - get_graph_stats()
  - get_room_topology()
  - get_room_with_devices()
  - get_scene_details()
  - get_scenes_by_mood()
  - get_scenes_for_room()
  - search_by_keywords()


In [ ]:
# Use the comprehensive room context query
# This is what GraphRAG will use!

query = queries.get_context_for_room_action()
results = conn.query(query, {"room_name": "Living"})

# Pretty print the context
import json
if results:
    context = results[0].get('context', results[0])
    print(json.dumps(context, indent=2, default=str))

CypherSyntaxError: {neo4j_code: Neo.ClientError.Statement.SyntaxError} {message: Aggregation column contains implicit grouping expressions. For example, in 'RETURN n.a, n.a + n.b + count(*)' the aggregation expression 'n.a + n.b + count(*)' includes the implicit grouping key 'n.b'. It may be possible to rewrite the query by extracting these grouping/aggregation expressions into a preceding WITH clause. Illegal expression(s): r.name, scenes, r.floor, r.type, devices (line 45, column 28 (offset: 1451))
"                    type: r.type,"
                            ^} {gql_status: 42001} {gql_status_description: error: syntax error or access rule violation - invalid syntax}

## 6. Visualizing the Graph

Let's create a simple visualization using NetworkX and Matplotlib.

In [ ]:
try:
    import networkx as nx
    import matplotlib.pyplot as plt
    VISUALIZATION_AVAILABLE = True
except ImportError:
    print("Install networkx and matplotlib for visualization:")
    print("  pip install networkx matplotlib")
    VISUALIZATION_AVAILABLE = False

In [ ]:
if VISUALIZATION_AVAILABLE:
    # Get graph data
    graph_data = conn.query("""
        MATCH (r:Room)-[:CONTAINS]->(d:Device)
        RETURN r.name AS room, d.name AS device, d.device_type AS device_type
    """)
    
    # Create NetworkX graph
    G = nx.Graph()
    
    # Add nodes and edges
    rooms = set()
    for row in graph_data:
        room = row['room']
        device = row['device']
        device_type = row['device_type']
        
        rooms.add(room)
        G.add_node(room, node_type='room')
        G.add_node(device, node_type=device_type)
        G.add_edge(room, device)
    
    # Create visualization
    plt.figure(figsize=(14, 10))
    
    # Color nodes by type
    color_map = []
    for node in G.nodes():
        node_type = G.nodes[node].get('node_type', 'unknown')
        if node_type == 'room':
            color_map.append('#4CAF50')  # Green for rooms
        elif node_type == 'light':
            color_map.append('#FFC107')  # Yellow for lights
        elif node_type == 'speaker':
            color_map.append('#2196F3')  # Blue for speakers
        elif node_type == 'television':
            color_map.append('#9C27B0')  # Purple for TV
        else:
            color_map.append('#9E9E9E')  # Gray for others
    
    # Layout and draw
    pos = nx.spring_layout(G, k=2, iterations=50)
    nx.draw(G, pos, 
            node_color=color_map,
            node_size=2000,
            font_size=8,
            font_weight='bold',
            with_labels=True,
            edge_color='#CCCCCC')
    
    plt.title("Smart Home Knowledge Graph: Rooms and Devices", fontsize=14)
    plt.tight_layout()
    plt.show()
    
    # Legend
    print("\nLegend:")
    print("  🟢 Green = Room")
    print("  🟡 Yellow = Light")
    print("  🔵 Blue = Speaker")
    print("  🟣 Purple = TV")
    print("  ⚪ Gray = Other")

## 7. Exercises

Try these queries on your own!

In [ ]:
# Exercise 1: Find all rooms on the "First Floor"
# Your code here:



In [ ]:
# Exercise 2: Find all devices that can play music
# Hint: Look for capability name 'play_music'
# Your code here:



In [ ]:
# Exercise 3: Which rooms are adjacent to the Kitchen?
# Hint: Use the ADJACENT_TO relationship
# Your code here:



In [ ]:
# Exercise 4: What scene has mood='relaxed'?
# Your code here:



## Summary

In this notebook, we learned:

1. **Connecting to Neo4j** - Using the Python driver and our connection wrapper
2. **Graph Schema** - Nodes (Room, Device, Capability, Scene) and Relationships (CONTAINS, HAS_CAPABILITY, etc.)
3. **Cypher Basics** - MATCH patterns, parameters, and relationship traversal
4. **Why Graphs?** - Intuitive queries for relationship-heavy data

Next: `02_graphrag_basics.ipynb` - Using the retriever for GraphRAG

In [ ]:
# Clean up
conn.close()
print("Connection closed.")